# 5 · Putting it together

⏱ about 6 minutes

You built every piece on its own. Here they are in one chatbot.

![The two halves of RAG: preparing the documents once, and answering every question](img/overview.svg)

| Piece | Notebook | In the app below |
|---|---|---|
| Read a PDF with Docling | 2 | `converter.convert()` in `add_pdf()` |
| Cut into chunks | 2 | `chunk()` |
| Turn text into vectors | 3 | `embed()` |
| Store and search in Qdrant | 3 | `store()`, `search()` |
| Answer from the chunks | 4 | `SYSTEM_PROMPT`, `on_message()` |
| Cite the page | 4 | the end of `on_message()` |
| Score the answer | 4 | `faithfulness()`, under every answer |

The app starts with the three papers from notebook 3. New is `add_pdf()`: attach a PDF to a
message and it runs the whole top row of the picture live, in the chat — read, chunk, embed,
store. One step for each, so you can watch it. Under every answer you now see its faithfulness
score: that is notebook 4's judge, built in.

In [ ]:
%%writefile app.py
import json
import os
import re

import chainlit as cl
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions
from docling.document_converter import DocumentConverter, PdfFormatOption
from dotenv import load_dotenv
from openai import AsyncOpenAI
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, PointStruct, VectorParams

load_dotenv()
llm = AsyncOpenAI(base_url=os.environ["LLM_BASE_URL"], api_key=os.environ["LLM_API_KEY"])
# These PDFs contain real text, so we skip OCR (reading text out of images).
converter = DocumentConverter(
    format_options={InputFormat.PDF: PdfFormatOption(pipeline_options=PdfPipelineOptions(do_ocr=False))}
)
SYSTEM_PROMPT = (
    "Answer the question using only the context below. "
    "If the context does not contain the answer, say that you don't know. "
    "After each statement, cite the context it comes from as [1], [2], …"
)


JUDGE = (
    "Split the answer into its individual factual claims. For each claim, decide whether the context supports it. "
    'Reply with JSON only, in this form: {"claims": [{"claim": "...", "supported": true}]}'
)


# Notebook 4: a second model call checks every claim against the chunks.
async def faithfulness(answer, context):
    reply = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"],
        messages=[
            {"role": "system", "content": JUDGE},
            {"role": "user", "content": f"Context:\n{context}\n\nAnswer:\n{answer}"},
        ],
        response_format={"type": "json_object"},
    )
    claims = json.loads(reply.choices[0].message.content)["claims"]
    if not claims:  # "I don't know" makes no claims, so there is nothing to check
        return None
    return sum(c["supported"] for c in claims) / len(claims)


# Notebook 2: cut a page into overlapping pieces.
def chunk(text, size=1000, overlap=200):
    starts = range(0, max(len(text) - overlap, 1), size - overlap)
    return [text[start : start + size] for start in starts]


# Notebook 3: text in, vectors out; question in, closest chunks out.
async def embed(texts):
    response = await llm.embeddings.create(model=os.environ["EMBED_MODEL"], input=texts)
    return [item.embedding for item in response.data]


async def search(question, limit=5):
    # Bonus: only search the file from cl.user_session, if there is one.
    return db.query_points("papers", query=(await embed([question]))[0], limit=limit).points


def store(chunks):
    first_id = db.count("papers").count
    db.upsert(
        "papers",
        points=[
            PointStruct(id=first_id + i, vector=c["vector"], payload={k: c[k] for k in ("source", "page", "text", "path")})
            for i, c in enumerate(chunks)
        ],
    )


papers = json.load(open("index.json"))
db = QdrantClient(":memory:")
db.create_collection(
    "papers",
    vectors_config=VectorParams(size=len(papers[0]["vector"]), distance=Distance.COSINE),
)
store([{**c, "path": f"data/{c['source']}"} for c in papers])


async def add_pdf(path, name):
    """The 'ahead of time' half for one new PDF: read, chunk, embed, store."""
    async with cl.Step(name=f"Docling reads {name}"):
        document = (await cl.make_async(converter.convert)(path)).document
    async with cl.Step(name="Cut into chunks"):
        new = [
            {"source": name, "page": number, "text": piece, "path": path}
            for number in document.pages
            for piece in chunk(document.export_to_markdown(page_no=number))
            if piece.strip()
        ]
    async with cl.Step(name="Embed and store in Qdrant"):
        for start in range(0, len(new), 64):
            batch = new[start : start + 64]
            for item, vector in zip(batch, await embed([c["text"] for c in batch])):
                item["vector"] = vector
        store(new)
    return len(new)


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])


# Notebook 4: search, answer, cite.
@cl.on_message
async def on_message(message: cl.Message):
    for file in message.elements:
        if file.name.lower().endswith(".pdf"):
            count = await add_pdf(file.path, file.name)
            await cl.Message(content=f"Added **{file.name}**: {count} chunks. Ask me about it.").send()
            # Bonus: remember the file name in cl.user_session.
    if not message.content.strip():
        return

    async with cl.Step(name="Search", type="retrieval") as step:
        hits = await search(message.content)
        step.output = "\n\n".join(
            f"**{h.payload['source']}, p. {h.payload['page']}** ({h.score:.2f})" for h in hits
        )

    context = "\n\n".join(f"[{i}] {h.payload['text']}" for i, h in enumerate(hits, start=1))
    history = cl.user_session.get("history")
    prompt = {"role": "user", "content": f"Context:\n{context}\n\nQuestion: {message.content}"}

    answer = cl.Message(content="")
    stream = await llm.chat.completions.create(
        model=os.environ["CHAT_MODEL"], messages=history + [prompt], stream=True
    )
    async for part in stream:
        if part.choices and part.choices[0].delta.content:
            await answer.stream_token(part.choices[0].delta.content)
    score = await faithfulness(answer.content, context)
    if score is not None:
        answer.content += f"\n\n*Faithfulness: {score:.2f}*"

    pdfs = {}
    for number in sorted({int(n) for n in re.findall(r"\[(\d+)\]", answer.content)}):
        if 1 <= number <= len(hits):
            hit = hits[number - 1].payload
            name = f"{hit['source'].removesuffix('.pdf')}, p. {hit['page']}"
            pdfs[name] = cl.Pdf(name=name, path=hit["path"], page=hit["page"], display="side")
    if pdfs:
        answer.content += "\n\n**Sources:** " + " · ".join(pdfs)
        answer.elements = list(pdfs.values())
    await answer.send()

    history.append({"role": "user", "content": message.content})
    history.append({"role": "assistant", "content": answer.content})

In [ ]:
import socket
import subprocess
import sys

# Start the chatbot, unless it's still running from an earlier notebook.
with socket.socket() as probe:
    running = probe.connect_ex(("127.0.0.1", 8000)) == 0
if not running:
    subprocess.Popen(
        [sys.executable, "-m", "chainlit", "run", "app.py", "--watch", "--headless"],
        stdout=open("chainlit.log", "w"),
        stderr=subprocess.STDOUT,
    )
print("Chatbot: http://localhost:8000")

## Your turn

Edit the `%%writefile app.py` cell above, run it again, and try your change in the browser.

1. **Your own document.** Attach a PDF of your own (a manual, a paper, a policy) and ask about it.
   Unfold the steps while it's being added. Does the source open the right page? What score do
   you get? Use test or public documents only: your PDF goes to the AISC gateway in Potsdam.
   A scanned PDF has no text layer, and with OCR switched off Docling reads nothing but image
   placeholders, so the bot has nothing to answer from. Worth knowing about your own collection.

**Bonus: chat with this document only.** Once you attach a PDF, the bot should answer from that PDF
only, not from the three papers. Remember the file name with `cl.user_session.set(...)` when it's
added, and give `search()` a Qdrant filter on `source`:
`query_filter=Filter(must=[FieldCondition(key="source", match=MatchValue(value=...))])`, all three
from `qdrant_client.models`.

### Before the round

Finish two sentences for yourself:

- *I'd use this for ...*
- *I expect ... to go wrong first, because ...*

<details>
<summary><b>Solutions</b> (try first!)</summary>

**Bonus** — `from qdrant_client.models import FieldCondition, Filter, MatchValue` at the top, then:
```python
            await cl.Message(content=f"Added **{file.name}**: {count} chunks. Ask me about it.").send()
            cl.user_session.set("document", file.name)
```
```python
async def search(question, limit=5):
    document = cl.user_session.get("document")
    only = Filter(must=[FieldCondition(key="source", match=MatchValue(value=document))]) if document else None
    return db.query_points("papers", query=(await embed([question]))[0], limit=limit, query_filter=only).points
```
</details>

## Questions people ask

**How does the score work?** It is `faithfulness()` near the top of the app, the same judge as in
notebook 4, written with `await` so the chat doesn't freeze while it runs. An "I don't know" gets no
score: it makes no claims the judge could check.

**Can I switch models and compare their scores?** Add a dropdown to the chat settings:
```python
from chainlit.input_widget import Select


@cl.on_chat_start
async def on_chat_start():
    cl.user_session.set("history", [{"role": "system", "content": SYSTEM_PROMPT}])
    await cl.ChatSettings([Select(id="model", label="Model", values=["gemma-4-31b", "gpt-oss-120b"], initial_index=0)]).send()


@cl.on_settings_update
async def on_settings_update(settings):
    cl.user_session.set("model", settings["model"])
```
and in `on_message`: `model=cl.user_session.get("model") or os.environ["CHAT_MODEL"]`. Comparing
models properly, over many questions, is what the template's evaluation app is for.

## Where to go from here

The full template this workshop is forked from (`apps/chainlit/`) runs the same steps, with
answers to the problems you met today:

| Today | The template |
|---|---|
| PDFs only | Also txt, md, csv and json |
| Fixed 1000-character chunks | Docling's structure-aware chunker and others, chosen per data source in a config file |
| Qdrant in memory, rebuilt on every start | Qdrant server, ingested once, updated when files change |
| Vector search only | Hybrid search: vectors plus keyword matching |
| `[1]` and a file name | Citation format set in the config: title, file, page |
| Faithfulness, one answer at a time | Faithfulness and relevance for every answer, and an app to compare configurations |